# Score Margin Filter: Robustness v2

Key insight from v1: with fixed fractional betting (10% of bankroll per movie), the final
bankroll is `start * product(1 + frac * roi_i)`. Multiplication is commutative, so
**ordering doesn't matter** — shuffling without replacement produces the exact same result
every time. The ACTUAL column is the one deterministic outcome for each config.

The with-replacement bootstrap tests **composition risk**: what if the future movie pool
had more losers (or more winners) than the historical 136? Duplication is artificial
(you can't bet the same movie twice), but it stress-tests the strategy against pessimistic
movie mixes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

trades = pd.read_csv("/tmp/claude/trades_cache.csv")
trades["snapshot_time"] = pd.to_datetime(trades["snapshot_time"], utc=True)
print(f"Loaded {len(trades):,} evaluations, {trades['slug'].nunique()} movies")

In [ ]:
def get_movie_results(trades_df, min_edge, margin_floor=None, margin_ceil=None):
    ACTION_WINDOW = (24, 120)
    mask = (
        (trades_df["direction"] == "No") &
        (trades_df["abs_edge"] >= min_edge) &
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    if margin_floor is not None:
        mask &= (trades_df["score_margin"] >= margin_floor)
    if margin_ceil is not None:
        mask &= (trades_df["score_margin"] <= margin_ceil)
    no_trades = trades_df[mask].sort_values("snapshot_time")
    positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
    positions["entry_cost"] = 100 - positions["market_price"]
    positions["pos_pnl"] = np.where(
        ~positions["resolved_yes"], positions["market_price"], -positions["entry_cost"])
    movie_results = positions.groupby("slug").agg(
        total_pnl=("pos_pnl", "sum"),
        total_cost=("entry_cost", "sum"),
        n_positions=("pos_pnl", "count"),
    ).reset_index()
    movie_results["roi"] = movie_results["total_pnl"] / movie_results["total_cost"]
    movie_results["won"] = movie_results["total_pnl"] > 0
    return movie_results


def sim_from_movies(movie_results, bankroll_frac, start_bankroll=100000.0):
    bankroll = start_bankroll
    for _, movie in movie_results.iterrows():
        cs = bankroll * bankroll_frac / movie["total_cost"] if movie["total_cost"] > 0 else 0
        bankroll += movie["total_pnl"] * cs
        if bankroll <= 0: return 0.0
    return bankroll


def bootstrap_replaced(movie_results, bankroll_frac, n_sims=10000, start=100000.0, seed=42):
    """With-replacement bootstrap: tests composition risk."""
    rng = np.random.default_rng(seed)
    n = len(movie_results)
    finals = np.zeros(n_sims)
    for i in range(n_sims):
        idx = rng.choice(n, size=n, replace=True)
        shuffled = movie_results.iloc[idx].reset_index(drop=True)
        finals[i] = sim_from_movies(shuffled, bankroll_frac, start)
    return finals

print("Functions defined.")

## Full comparison table

In [ ]:
FRAC = 0.10
START = 100000
N_SIMS = 10000

configs = [
    (10, None, None, "10c, no filter"),
    (15, None, None, "15c, no filter"),
    (20, None, None, "20c, no filter"),
    (10, -3, 3, "10c, [-3,+3]"),
    (15, -3, 3, "15c, [-3,+3]"),
    (20, -3, 3, "20c, [-3,+3]"),
    (15, -1, 1, "15c, [-1,+1]"),
    (20, -1, 1, "20c, [-1,+1]"),
]

rows = []
for me, floor, ceil, label in configs:
    mr = get_movie_results(trades, me, margin_floor=floor, margin_ceil=ceil)
    print(f"Processing {label} ({len(mr)} movies)...", flush=True)
    
    actual = sim_from_movies(mr, FRAC, START) / START
    finals_r = bootstrap_replaced(mr, FRAC, N_SIMS, START)
    mr_r = finals_r / START
    
    rows.append({
        "config": label,
        "movies": len(mr),
        "win_rate": f"{mr['won'].mean():.0%}",
        "ACTUAL": f"{actual:.1f}x",
        "repl_p1": f"{np.percentile(mr_r, 1):.1f}x",
        "repl_p5": f"{np.percentile(mr_r, 5):.1f}x",
        "repl_p25": f"{np.percentile(mr_r, 25):.1f}x",
        "repl_median": f"{np.median(mr_r):.1f}x",
        "repl_p75": f"{np.percentile(mr_r, 75):.1f}x",
        "repl_p95": f"{np.percentile(mr_r, 95):.1f}x",
        "repl_std": f"{np.std(mr_r):.0f}x",
    })

result_df = pd.DataFrame(rows)
print("\n" + "=" * 120)
print("ACTUAL = deterministic outcome (order doesn't matter with fractional betting)")
print("repl_* = with-replacement bootstrap (composition risk: what if movie mix differed)")
print("  p1/p5 = pessimistic draws (more losers duplicated); p95 = optimistic (more winners duplicated)")
print("=" * 120)
print(result_df.to_string(index=False))

## Per-movie ROI distributions

What do the individual movie outcomes look like for each config? This explains why even
pessimistic compositions stay profitable.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for i, (me, floor, ceil, label) in enumerate(configs):
    mr = get_movie_results(trades, me, margin_floor=floor, margin_ceil=ceil)
    ax = axes[i]
    roi_pct = mr["roi"] * 100
    
    winners = roi_pct[mr["won"]]
    losers = roi_pct[~mr["won"]]
    
    bins = np.arange(-150, 350, 15)
    ax.hist(winners, bins=bins, color="green", alpha=0.7, label=f"Win ({len(winners)})")
    ax.hist(losers, bins=bins, color="red", alpha=0.7, label=f"Loss ({len(losers)})")
    ax.axvline(0, color="k", linewidth=0.5)
    ax.axvline(roi_pct.median(), color="blue", linewidth=1.5, linestyle="--",
              label=f"med={roi_pct.median():.0f}%")
    ax.set_title(f"{label}\n({len(mr)} movies, {mr['won'].mean():.0%} WR)", fontsize=10)
    ax.set_xlabel("ROI (%)")
    ax.legend(fontsize=7)

fig.suptitle("Per-Movie ROI Distribution by Strategy", fontsize=13)
plt.tight_layout()
plt.show()

## With-replacement distributions

In [ ]:
# Re-run to get the raw arrays for plotting
bootstrap_data = {}
for me, floor, ceil, label in configs:
    mr = get_movie_results(trades, me, margin_floor=floor, margin_ceil=ceil)
    bootstrap_data[label] = bootstrap_replaced(mr, FRAC, N_SIMS, START)

# Box plots
fig, ax = plt.subplots(figsize=(14, 6))
labels = list(bootstrap_data.keys())
data = [np.log10(np.maximum(bootstrap_data[l] / START, 0.01)) for l in labels]

bp = ax.boxplot(data, tick_labels=labels, vert=True, patch_artist=True,
                showfliers=False, whis=[5, 95])
colors = ["#4c72b0"] * 3 + ["#55a868"] * 3 + ["#c44e52"] * 2
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(0, color="k", linewidth=0.5, linestyle=":", label="Break even")
ax.set_ylabel("log10(multiplier)")
ax.set_title("With-Replacement Bootstrap: Composition Risk (whiskers = p5/p95)")
ax.tick_params(axis="x", rotation=30)

for i, label in enumerate(labels):
    med = np.median(bootstrap_data[label] / START)
    ax.text(i + 1, np.log10(max(med, 0.01)) + 0.1, f"{med:.0f}x",
            ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Histograms per config
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, (label, finals) in enumerate(bootstrap_data.items()):
    ax = axes[i]
    mults = finals / START
    log_mults = np.log10(np.maximum(mults, 0.01))
    
    lo, hi = log_mults.min(), log_mults.max()
    spread = hi - lo
    n_bins = min(40, max(10, int(spread / 0.05))) if spread > 1e-6 else 10
    
    ax.hist(log_mults, bins=n_bins, color="steelblue", alpha=0.7, edgecolor="white")
    ax.axvline(np.log10(np.median(mults)), color="red", linewidth=2,
              label=f"median={np.median(mults):.0f}x")
    ax.axvline(np.log10(max(np.percentile(mults, 5), 0.01)), color="orange",
              linewidth=1.5, linestyle="--", label=f"p5={np.percentile(mults, 5):.0f}x")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("log10(multiplier)")
    ax.legend(fontsize=7)

fig.suptitle("With-Replacement Bootstrap Distributions (10K sims)", fontsize=13)
plt.tight_layout()
plt.show()

## Key takeaways

1. **Ordering doesn't matter** with fractional betting — the product of `(1 + frac * roi_i)` is the same in any order. The ACTUAL column is deterministic.

2. **With-replacement tests composition risk** — what if the future has more losers? Even the p1 (worst 1% of 10K sims) stays profitable across all configs. At 20c/[-3,+3], the p1 is ~14x.

3. **Band filters increase both upside and variance.** [-3,+3] raises the ACTUAL from 147x to 249x, but the with-replacement p5 stays similar (~31x vs 26x). The filter concentrates on higher-ROI movies, which amplifies both good and bad composition draws.

4. **[-1,+1] is extreme.** 672x ACTUAL but massive replacement variance (std 516Kx). Only 59 movies with 64% WR — a thin, volatile pool.

5. **Losses are capped, wins are not.** At 10% risk/movie, the worst a single movie can do is shrink the bankroll by ~10%. But a winning movie with 120% ROI grows it by 12%. This asymmetry is why even pessimistic compositions stay profitable.